# Sionna 0.19 Scene Builder

Portable scene builder using:
- **AWS Terrain Tiles** (elevation-tiles-prod, GeoTIFF, zoom 14 ≈ 2.4 m/px) — no spikes, global coverage, free, anonymous
- **OSM via osmnx** — buildings + roads + water + vegetation
- **Approach from sionna-large-radio-maps** (NVIDIA) adapted for Sionna 0.19 / no drjit dependency

### What this notebook does
1. Downloads AWS elevation tiles for the scene bbox → builds a clean heightmap
2. Downloads OSM buildings → extrudes each building with correct `base_z` from heightmap
3. Builds terrain PLY mesh with real DEM elevation
4. Writes `scene.xml` (Mitsuba 2.1.0) compatible with Sionna 0.19
5. Saves everything to `BASE_DIR/scene/` ready for the main simulation notebook

### Run order
`CELL 0 → CELL 1 → CELL 2 → CELL 3 → CELL 4 → CELL 5 → CELL 6`

In [ ]:
# ============================================================
# CELL 0 — CONFIG  (edit this cell only)
# ============================================================
import os

# ── Scene bbox (WGS84) ────────────────────────────────────────────────────
SCENE_WEST   = -1.409134
SCENE_EAST   = -1.245094
SCENE_SOUTH  =  52.910276
SCENE_NORTH  =  53.009090

# ── Output directory ─────────────────────────────────────────────────────
# BASE_DIR is detected from sionna019_main_simulation.ipynb if it exists
# in the same directory, otherwise falls back to the default path.
# SCENE_DIR is always BASE_DIR/scene/ to match the main notebook's expectation.
_this_dir = os.path.dirname(os.path.abspath('sionna019_scene_builder.ipynb'))
_main_nb  = os.path.join(_this_dir, 'sionna019_main_simulation.ipynb')

BASE_DIR = None
if os.path.exists(_main_nb):
    try:
        import json as _json
        _nb = _json.load(open(_main_nb))
        for _cell in _nb['cells']:
            _src = ''.join(_cell['source'])
            for _line in _src.split('\n'):
                if 'BASE_DIR' in _line and 'expanduser' in _line and '=' in _line and not _line.strip().startswith('#'):
                    _val = _line.split('=',1)[1].strip()
                    # eval only simple expanduser calls
                    if 'expanduser' in _val and 'FYP2026' in _val:
                        try:
                            BASE_DIR = eval(_val.split('os.path.expanduser')[1].replace('(','',1)
                                           .rsplit(')',1)[0], {'os': os, '__builtins__': {}})
                            BASE_DIR = os.path.expanduser(BASE_DIR)
                        except:
                            pass
                    if BASE_DIR:
                        break
            if BASE_DIR:
                break
        if BASE_DIR:
            print(f'Detected BASE_DIR from main notebook: {BASE_DIR}')
    except Exception as _e:
        print(f'Could not parse main notebook: {_e}')

if not BASE_DIR:
    BASE_DIR = os.path.expanduser('~/Documents/FYP2026/nottingham_11km')
    print(f'Using default BASE_DIR: {BASE_DIR}')

# SCENE_DIR always matches main notebook expectation: BASE_DIR/scene/
SCENE_DIR = os.path.join(BASE_DIR, 'scene')
MESH_DIR  = os.path.join(SCENE_DIR, 'meshes')
os.makedirs(MESH_DIR, exist_ok=True)
print(f'SCENE_DIR : {SCENE_DIR}')
print(f'MESH_DIR  : {MESH_DIR}')

# ── Coordinate system ────────────────────────────────────────────────────
UTM_EPSG  = 32630   # UTM zone 30N (covers UK)

# ── AWS tile zoom level ──────────────────────────────────────────────────
# z=14 → ~2.4 m/px at UK latitude (512×512 px tiles)
# z=13 → ~4.8 m/px  (faster download, less detail)
TILE_ZOOM  = 14

# ── Terrain mesh resolution ───────────────────────────────────────────────
# Number of grid points per axis for the terrain PLY.
# 500 → 500×500 = 250k verts, ~500k triangles (~10 MB PLY) — recommended
# 200 → 200×200 = 40k  verts, ~80k triangles  (~1.5 MB PLY) — fast
TERRAIN_GRID_N = 500

# ── Building parameters ───────────────────────────────────────────────────
MIN_BUILDING_AREA_M2  = 30.0   # skip footprints smaller than this
CITY_MIN_HEIGHT_M     =  2.0   # clamp building height to at least this
CITY_MAX_HEIGHT_M     = 40.0   # clamp building height to at most this
HEIGHT_PER_LEVEL_M    =  3.5   # used when only building:levels tag present
DEFAULT_HEIGHT_M      =  8.0   # fallback when no height/levels tag

# ── OSM extra features ────────────────────────────────────────────────────
INCLUDE_ROADS       = True
INCLUDE_WATER       = True
INCLUDE_VEGETATION  = False   # polygon veg — adds clutter for macro-cell sim

# ── Overture Maps building height enrichment ─────────────────────────────────
# When True: queries Overture Maps (AWS S3) to fill missing building heights
# where OSM has no height/building:levels tag (currently fall back to 8m default).
# Requires: pip install overturemaps pyarrow
# Set False to skip and use OSM-only heights (faster, no extra dependency).
USE_OVERTURE_HEIGHTS = True

# ── Meta AI Global Canopy Height Model (CHM) ─────────────────────────────────
# When True: downloads Meta/WRI 1m canopy height tiles from AWS S3 (free, no
# credentials) and adds tree cylinders to the scene for urban/park vegetation.
# S3: s3://dataforgood-fb-data/forests/v1/alsgedi_global_v6_float/chm/
# Requires: pip install rasterio (already needed for DEM tiles)
# Set False to skip (faster, no tree geometry in scene).
USE_CANOPY_HEIGHTS   = True

# Tree geometry parameters
TREE_MIN_HEIGHT_M    = 3.0    # ignore canopy pixels below this (noise/shrubs)
TREE_MAX_HEIGHT_M    = 30.0   # cap on individual tree height
TREE_RADIUS_M        = 3.0    # cylinder radius for each tree (approx crown)
TREE_GRID_SPACING_M  = 5.0    # sample canopy every N metres (avoids too many cylinders)
TREE_MATERIAL        = 'itu_vegetation'   # ITU-R P.833 vegetation material

# ── Terrain material ──────────────────────────────────────────────────────────
# itu_wet_ground    : eps=30,  sigma=0.02  → very high reflectivity, creates
#                    thousands of specular ground-bounce paths → RSSI too high
# itu_urban_ground  : eps=5.0, sigma=0.50  → absorbing concrete/asphalt mix,
#                    breaks long-range specular bounces (recommended for urban)
# itu_concrete      : eps=5.31,sigma=0.092 → moderate, use for suburban
TERRAIN_MATERIAL    = 'itu_urban_ground'   # custom absorbing urban ground

# ── Exclude small/irrelevant building types ───────────────────────────────
EXCLUDE_BUILDING_TYPES = {
    'garage','garages','carport','shed','hut','roof','canopy',
    'kiosk','bicycle_parking','service','greenhouse','barn',
    'stable','sty','storage_tank','container','tent',
    'grandstand','shelter','utility','gatehouse',
}

print('Config loaded.')
print(f'  Scene bbox  : lon [{SCENE_WEST}, {SCENE_EAST}]')
print(f'                lat [{SCENE_SOUTH}, {SCENE_NORTH}]')
print(f'  Output      : {SCENE_DIR}')
print(f'  Tile zoom   : {TILE_ZOOM}  (~{156543.03 * np.cos(np.radians((SCENE_SOUTH+SCENE_NORTH)/2)) / 2**TILE_ZOOM:.1f} m/px)' if 'np' in dir() else f'  Tile zoom   : {TILE_ZOOM}')

In [ ]:
# ============================================================
# CELL 1 — IMPORTS & DEPENDENCIES
# ============================================================
import os, math, time, json, struct, warnings
import numpy as np
import requests
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor, as_completed
from pyproj import Transformer
import shapely.geometry as sg
import shapely.ops as so
from shapely.geometry import Polygon, MultiPolygon, box

# Optional: rasterio for GeoTIFF reading
try:
    import rasterio
    from rasterio.transform import from_bounds
    _HAS_RASTERIO = True
except ImportError:
    _HAS_RASTERIO = False
    print('⚠  rasterio not found — falling back to PIL for GeoTIFF tiles')

# PIL for fallback
try:
    from PIL import Image
    _HAS_PIL = True
except ImportError:
    _HAS_PIL = False

# osmnx for OSM data
try:
    import osmnx as ox
    ox.settings.use_cache = True
    ox.settings.log_console = False
    _HAS_OSMNX = True
except ImportError:
    _HAS_OSMNX = False
    print('⚠  osmnx not found — install with: pip install osmnx')

# trimesh for PLY export
try:
    import trimesh
    _HAS_TRIMESH = True
except ImportError:
    _HAS_TRIMESH = False
    print('⚠  trimesh not found — install with: pip install trimesh')

# Coordinate transformers
to_utm   = Transformer.from_crs('EPSG:4326', f'EPSG:{UTM_EPSG}', always_xy=True)
to_wgs84 = Transformer.from_crs(f'EPSG:{UTM_EPSG}', 'EPSG:4326', always_xy=True)

# Scene bbox in UTM
sw_utm = to_utm.transform(SCENE_WEST,  SCENE_SOUTH)
ne_utm = to_utm.transform(SCENE_EAST,  SCENE_NORTH)
center_utm = ((sw_utm[0]+ne_utm[0])/2, (sw_utm[1]+ne_utm[1])/2)
center_lon, center_lat = to_wgs84.transform(*center_utm)

print(f'UTM SW : {sw_utm[0]:.1f}, {sw_utm[1]:.1f}')
print(f'UTM NE : {ne_utm[0]:.1f}, {ne_utm[1]:.1f}')
print(f'Center : ({center_lon:.5f}, {center_lat:.5f})')
print(f'Size   : {(ne_utm[0]-sw_utm[0])/1000:.2f} km × {(ne_utm[1]-sw_utm[1])/1000:.2f} km')

In [ ]:
# ============================================================
# CELL 2 — AWS ELEVATION TILES → HEIGHTMAP
# ============================================================
# Downloads GeoTIFF tiles from the public AWS elevation-tiles-prod bucket
# (same source used by Mapzen/Terrarium, Cesium, sionna-large-radio-maps).
# No credentials needed — anonymous public access.
# URL: s3://elevation-tiles-prod/geotiff/{z}/{x}/{y}.tif
# Values are direct metres ASL (float32 GeoTIFF, no conversion formula).

AWS_BASE = 'https://s3.amazonaws.com/elevation-tiles-prod/geotiff'

def _lon2tile(lon, z):
    return int(math.floor((lon + 180) / 360 * 2**z))

def _lat2tile(lat, z):
    lat_r = math.radians(lat)
    return int(math.floor((1 - math.log(math.tan(lat_r) + 1/math.cos(lat_r)) / math.pi) / 2 * 2**z))

def _tile2lon(x, z):
    return x / 2**z * 360 - 180

def _tile2lat(y, z):
    n = math.pi - 2 * math.pi * y / 2**z
    return math.degrees(math.atan(math.sinh(n)))

def _download_tile(z, x, y, retries=3):
    url = f'{AWS_BASE}/{z}/{x}/{y}.tif'
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
            buf = BytesIO(r.content)
            if _HAS_RASTERIO:
                with rasterio.open(buf) as ds:
                    data = ds.read(1).astype(np.float32)
            elif _HAS_PIL:
                # PIL may not read float GeoTIFF correctly — warn
                img = Image.open(buf)
                data = np.array(img, dtype=np.float32)
            else:
                raise RuntimeError('Need rasterio or PIL to read GeoTIFF tiles')
            # Resample to 512×512 if needed
            if data.shape != (512, 512):
                from scipy.ndimage import zoom as nd_zoom
                fx = 512 / data.shape[0]; fy = 512 / data.shape[1]
                data = nd_zoom(data, (fx, fy), order=1).astype(np.float32)
            return data
        except Exception as e:
            if attempt == retries - 1:
                print(f'  ⚠ tile {z}/{x}/{y} failed: {e} — using zeros')
                return np.zeros((512, 512), dtype=np.float32)
            time.sleep(2 ** attempt)

# Compute tile range for scene bbox
x0 = _lon2tile(SCENE_WEST,  TILE_ZOOM)
x1 = _lon2tile(SCENE_EAST,  TILE_ZOOM)
y0 = _lat2tile(SCENE_NORTH, TILE_ZOOM)  # north = smaller y in tile coords
y1 = _lat2tile(SCENE_SOUTH, TILE_ZOOM)
n_cols = x1 - x0 + 1
n_rows = y1 - y0 + 1
print(f'Tile range : x=[{x0},{x1}] y=[{y0},{y1}]  →  {n_cols}×{n_rows} = {n_cols*n_rows} tiles')

# Download in parallel
tile_mosaic = np.zeros((n_rows * 512, n_cols * 512), dtype=np.float32)
jobs = [(TILE_ZOOM, x0+col, y0+row, col, row)
        for row in range(n_rows) for col in range(n_cols)]

print(f'Downloading {len(jobs)} tiles ...')
t0 = time.time()
with ThreadPoolExecutor(max_workers=8) as ex:
    futures = {ex.submit(_download_tile, z, x, y): (col, row)
               for z, x, y, col, row in jobs}
    for i, fut in enumerate(as_completed(futures)):
        col, row = futures[fut]
        tile_mosaic[row*512:(row+1)*512, col*512:(col+1)*512] = fut.result()
        if (i+1) % max(1, len(jobs)//4) == 0:
            print(f'  {i+1}/{len(jobs)} tiles done')

print(f'Download done in {time.time()-t0:.1f}s')

# Extent of the mosaic in WGS84
_map_min_lon = _tile2lon(x0,   TILE_ZOOM)
_map_max_lon = _tile2lon(x1+1, TILE_ZOOM)
_map_max_lat = _tile2lat(y0,   TILE_ZOOM)   # y0 is north
_map_min_lat = _tile2lat(y1+1, TILE_ZOOM)   # y1+1 is south
_mosaic_h, _mosaic_w = tile_mosaic.shape

print(f'Mosaic size : {_mosaic_w}×{_mosaic_h} px')
print(f'Mosaic lon  : [{_map_min_lon:.5f}, {_map_max_lon:.5f}]')
print(f'Mosaic lat  : [{_map_min_lat:.5f}, {_map_max_lat:.5f}]')
print(f'Elevation   : [{tile_mosaic.min():.1f}, {tile_mosaic.max():.1f}] m ASL')

def height_from_wgs84(lon, lat):
    """Bilinear interpolation from mosaic. Returns elevation in metres ASL."""
    u = (lon - _map_min_lon) / (_map_max_lon - _map_min_lon) * (_mosaic_w - 1)
    v = (1 - (lat - _map_min_lat) / (_map_max_lat - _map_min_lat)) * (_mosaic_h - 1)
    u = np.clip(u, 0, _mosaic_w - 1)
    v = np.clip(v, 0, _mosaic_h - 1)
    x0i, y0i = int(np.floor(u)), int(np.floor(v))
    x1i, y1i = min(x0i+1, _mosaic_w-1), min(y0i+1, _mosaic_h-1)
    fx, fy = u - x0i, v - y0i
    h = (tile_mosaic[y0i, x0i] * (1-fx) * (1-fy)
       + tile_mosaic[y0i, x1i] *    fx  * (1-fy)
       + tile_mosaic[y1i, x0i] * (1-fx) *    fy
       + tile_mosaic[y1i, x1i] *    fx  *    fy)
    return float(h)

def height_from_utm(easting, northing):
    """Height at UTM coordinates."""
    lon, lat = to_wgs84.transform(easting, northing)
    return height_from_wgs84(lon, lat)

# Scene centre elevation = local z=0 reference
origin_elev_asl = height_from_wgs84(center_lon, center_lat)
print(f'\nScene centre elevation : {origin_elev_asl:.2f} m ASL  (= local z=0)')

def local_z(lon, lat):
    """Returns local z in metres relative to scene centre elevation."""
    return height_from_wgs84(lon, lat) - origin_elev_asl

In [ ]:
# ============================================================
# CELL 3 — BUILD TERRAIN PLY
# ============================================================
# Generates a regular grid terrain mesh with DEM elevation.
# Building footprints are NOT cut out (no tool does this —
# buildings are separate meshes placed on the terrain surface).

N = TERRAIN_GRID_N
print(f'Building terrain mesh: {N}×{N} grid ({2*(N-1)**2:,} triangles) ...')

# UTM grid spanning scene bbox, centred at (0,0)
x_span = ne_utm[0] - sw_utm[0]
y_span = ne_utm[1] - sw_utm[1]
xs = np.linspace(-x_span/2, x_span/2, N, dtype=np.float32)
ys = np.linspace(-y_span/2, y_span/2, N, dtype=np.float32)
XX, YY = np.meshgrid(xs, ys, indexing='ij')   # shape (N, N)

# DEM elevation for each grid point
ZZ = np.zeros((N, N), dtype=np.float32)
t0 = time.time()
for i in range(N):
    for j in range(N):
        utm_x = center_utm[0] + XX[i, j]
        utm_y = center_utm[1] + YY[i, j]
        ZZ[i, j] = height_from_utm(utm_x, utm_y) - origin_elev_asl
    if (i+1) % max(1, N//5) == 0:
        print(f'  row {i+1}/{N}  ({time.time()-t0:.0f}s)')

print(f'DEM range: [{ZZ.min():.1f}, {ZZ.max():.1f}] m (local)')

# Build vertex array and face array
verts = np.stack([XX.ravel(), YY.ravel(), ZZ.ravel()], axis=1)  # (N*N, 3)

faces = []
for i in range(N-1):
    for j in range(N-1):
        a = i*N + j
        b = a + 1
        c = a + N
        d = c + 1
        faces.append([a, b, d])
        faces.append([a, d, c])
faces = np.array(faces, dtype=np.int32)
print(f'Vertices: {len(verts):,}  Faces: {len(faces):,}')

# Export terrain PLY
terrain_ply = os.path.join(MESH_DIR, 'terrain.ply')
if _HAS_TRIMESH:
    mesh = trimesh.Trimesh(vertices=verts, faces=faces, process=False)
    mesh.export(terrain_ply)
else:
    # Minimal ASCII PLY writer
    with open(terrain_ply, 'w') as f:
        f.write('ply\nformat ascii 1.0\n')
        f.write(f'element vertex {len(verts)}\n')
        f.write('property float x\nproperty float y\nproperty float z\n')
        f.write(f'element face {len(faces)}\n')
        f.write('property list uchar int vertex_indices\nend_header\n')
        for v in verts:
            f.write(f'{v[0]:.4f} {v[1]:.4f} {v[2]:.4f}\n')
        for face in faces:
            f.write(f'3 {face[0]} {face[1]} {face[2]}\n')

sz = os.path.getsize(terrain_ply)/1024
print(f'Saved: {terrain_ply}  ({sz:.0f} KB)')

In [ ]:
# ============================================================
# CELL 4 - OSM BUILDINGS -> MERGED PLYs BY MATERIAL
# ============================================================
# Accumulates all wall/roof geometry per material in memory,
# then writes ONE merged PLY per (role, material) combination.
# Result: ~10 PLY files + terrain instead of ~95k individual files.

assert _HAS_OSMNX, 'osmnx required: pip install osmnx'

print('Downloading OSM buildings ...')
t0 = time.time()
try:
    # osmnx >=2.0: bbox=(west, south, east, north)
    gdf_bld = ox.features_from_bbox(
        bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH),
        tags={'building': True}
    )
except Exception as e:
    print(f'osmnx error: {e}')
    raise
print(f'  {len(gdf_bld)} raw building features  ({time.time()-t0:.1f}s)')

# ── Overture Maps height enrichment ───────────────────────────────────────────
# Downloads building heights from Overture Maps for footprints where OSM
# has no height/building:levels tag. Uses spatial join on centroid.
_overture_heights = {}   # lon,lat centroid -> height_m

if globals().get('USE_OVERTURE_HEIGHTS', False):
    try:
        import overturemaps
        import pyarrow as pa
        print('Fetching Overture Maps building heights ...')
        t_ov = time.time()
        _ov = overturemaps.record_batch_reader(
            'building',
            bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH)
        )
        _ov_table = _ov.read_all()
        _ov_df = _ov_table.to_pandas()
        print(f'  Overture: {len(_ov_df)} buildings  ({time.time()-t_ov:.1f}s)')

        # Extract centroid lon/lat and height
        _n_heights = 0
        for _, _row in _ov_df.iterrows():
            _h = _row.get('height', None)
            if _h is None or (isinstance(_h, float) and np.isnan(_h)):
                continue
            try:
                _h = float(_h)
                if _h < 1 or _h > 200:
                    continue
                _geom = _row.get('geometry', None)
                if _geom is None:
                    continue
                import shapely.wkb as _swkb
                _g = _swkb.loads(_geom) if isinstance(_geom, bytes) else _geom
                _cx = round(_g.centroid.x, 5)
                _cy = round(_g.centroid.y, 5)
                _overture_heights[(_cx, _cy)] = _h
                _n_heights += 1
            except:
                continue
        print(f'  Overture heights loaded: {_n_heights}')
    except ImportError:
        print('  overturemaps not installed — skipping (pip install overturemaps pyarrow)')
    except Exception as _e:
        print(f'  Overture fetch failed: {_e} — using OSM heights only')

# NaN-safe tag reader: pandas returns float NaN for missing OSM columns
def _tag(row, key):
    v = row.get(key, '')
    if v is None: return ''
    s = str(v).strip().lower()
    return '' if s in ('nan', 'none', 'no') else s

def _bld_mat(row):
    mat  = _tag(row, 'building:material')
    fmat = _tag(row, 'building:facade:material')
    tag  = _tag(row, 'building')
    amen = _tag(row, 'amenity')
    shop = _tag(row, 'shop')
    off  = _tag(row, 'office')
    if 'glass' in mat or 'glass' in fmat:                return 'itu_glass'
    if 'wood'  in mat or 'timber' in mat:                return 'itu_wood'
    if 'wood'  in fmat or 'timber' in fmat:              return 'itu_wood'
    if 'brick' in mat or 'brick' in fmat:                return 'itu_brick'
    if 'stone' in mat or 'stone' in fmat:                return 'itu_brick'
    if tag in ('greenhouse','glasshouse'):               return 'itu_glass'
    if amen in ('shopping_centre','mall'):               return 'itu_glass'
    if shop in ('mall','supermarket','department_store'):return 'itu_glass'
    if off and off not in ('yes','true','1'):            return 'itu_glass'
    if tag in ('residential','house','detached','semidetached_house',
               'semi_detached','terrace','terrace_house','bungalow',
               'farm','farmhouse','dormitory','apartments','block'):
        return 'itu_brick'
    if tag in ('industrial','warehouse','factory','shed',
               'storage_tank','silo','barn'):            return 'itu_concrete'
    if tag in ('retail','commercial','supermarket','kiosk'): return 'itu_glass'
    if tag in ('cathedral','church','chapel','mosque','temple'): return 'itu_brick'
    if tag in ('school','university','hospital','civic','public'): return 'itu_concrete'
    return 'itu_brick'

def _roof_mat(row):
    roof_tag = _tag(row, 'roof:material')
    btag     = _tag(row, 'building')
    if any(k in roof_tag for k in ['metal','steel','zinc','aluminium','copper','tin']):
        return 'itu_metal'
    if 'glass' in roof_tag:   return 'itu_glass'
    if any(k in roof_tag for k in ['wood','timber','thatch']): return 'itu_wood'
    if any(k in roof_tag for k in ['tile','concrete','slate','terracotta']): return 'itu_concrete'
    if btag in ('industrial','warehouse','factory','shed','barn',
                'retail','supermarket','commercial','garage','garages'):
        return 'itu_metal'
    return 'itu_concrete'

def _bld_height(row, centroid_lon=None, centroid_lat=None):
    # 1. OSM height tag
    try:
        h = float(str(row.get('height','0')).replace('m','').strip())
        if h > 1: return float(np.clip(h, CITY_MIN_HEIGHT_M, CITY_MAX_HEIGHT_M))
    except: pass
    # 2. OSM building:levels tag
    try:
        lvl = float(str(row.get('building:levels','0')).strip())
        if lvl > 0: return float(np.clip(lvl * HEIGHT_PER_LEVEL_M, CITY_MIN_HEIGHT_M, CITY_MAX_HEIGHT_M))
    except: pass
    # 3. Overture Maps height (nearest centroid match within 10m)
    if centroid_lon is not None and _overture_heights:
        _key = (round(centroid_lon, 5), round(centroid_lat, 5))
        if _key in _overture_heights:
            return float(np.clip(_overture_heights[_key], CITY_MIN_HEIGHT_M, CITY_MAX_HEIGHT_M))
    # 4. Default fallback
    return DEFAULT_HEIGHT_M

# Accumulators: (role, mat) -> {V: list of arrays, F: list of arrays, offset: int}
_accum = {}

def _add_mesh(role, mat, verts, faces):
    key = (role, mat)
    if key not in _accum:
        _accum[key] = {'V': [], 'F': [], 'offset': 0}
    a = _accum[key]
    a['F'].append(faces + a['offset'])
    a['V'].append(verts)
    a['offset'] += len(verts)

def _extrude_walls(pts, base_z, height):
    n   = len(pts)
    top = base_z + height
    V, F = [], []
    for i in range(n):
        j  = (i+1) % n
        vb = len(V)
        V += [[pts[i,0], pts[i,1], base_z],
              [pts[j,0], pts[j,1], base_z],
              [pts[j,0], pts[j,1], top   ],
              [pts[i,0], pts[i,1], top   ]]
        F += [[vb, vb+1, vb+2], [vb, vb+2, vb+3]]
    return np.array(V, np.float32), np.array(F, np.int32)

def _extrude_roof(pts, top_z):
    cx, cy = pts[:,0].mean(), pts[:,1].mean()
    n = len(pts)
    V = [[p[0], p[1], top_z] for p in pts] + [[cx, cy, top_z]]
    ci = n
    F = []
    for i in range(n):
        j = (i+1) % n
        a = np.array([*V[i][:2], 0.0]) - np.array([*V[ci][:2], 0.0])
        b = np.array([*V[j][:2], 0.0]) - np.array([*V[ci][:2], 0.0])
        if abs(float(np.cross(a, b)[2])) > 1e-6:
            F.append([i, j, ci])
    if not F:
        return None, None
    return np.array(V, np.float32), np.array(F, np.int32)

# Process buildings
n_ok = n_skip = 0
for oid, row in gdf_bld.iterrows():
    geom = row.geometry
    if geom is None or geom.is_empty:
        n_skip += 1; continue
    if isinstance(geom, MultiPolygon):
        geom = max(geom.geoms, key=lambda g: g.area)
    if not isinstance(geom, Polygon):
        n_skip += 1; continue

    btag = str(row.get('building','')).lower()
    if btag in EXCLUDE_BUILDING_TYPES:
        n_skip += 1; continue

    coords_utm = []
    for lon, lat in geom.exterior.coords:
        ex, ny = to_utm.transform(lon, lat)
        coords_utm.append((ex - center_utm[0], ny - center_utm[1]))

    if sg.Polygon(coords_utm).area < MIN_BUILDING_AREA_M2:
        n_skip += 1; continue

    pts = np.array(coords_utm, np.float32)
    if np.allclose(pts[0], pts[-1]): pts = pts[:-1]
    if len(pts) < 3: n_skip += 1; continue

    base_z = local_z(geom.centroid.x, geom.centroid.y)
    h      = _bld_height(row, geom.centroid.x, geom.centroid.y)
    w_mat  = _bld_mat(row)
    r_mat  = _roof_mat(row)

    wV, wF = _extrude_walls(pts, base_z, h)
    _add_mesh('wall', w_mat, wV, wF)

    rV, rF = _extrude_roof(pts, base_z + h)
    if rV is not None:
        _add_mesh('roof', r_mat, rV, rF)

    n_ok += 1
    if n_ok % 5000 == 0:
        print(f'  {n_ok} buildings processed ...')

print(f'Buildings  : {n_ok} processed, {n_skip} skipped')

# Remove old individual PLYs if any
import glob as _glob
for _f in _glob.glob(os.path.join(MESH_DIR, 'bld_*.ply')):
    os.remove(_f)

# Write one merged PLY per (role, material)
mat_plys = {}
total_verts = total_faces = 0
for (role, mat), a in sorted(_accum.items()):
    V = np.concatenate(a['V'], axis=0).astype(np.float32)
    F = np.concatenate(a['F'], axis=0).astype(np.int32)
    fname = f'{role}_{mat}.ply'
    fpath = os.path.join(MESH_DIR, fname)
    if _HAS_TRIMESH:
        trimesh.Trimesh(vertices=V, faces=F, process=False).export(fpath)
    else:
        with open(fpath, 'w') as f:
            f.write('ply\nformat ascii 1.0\n')
            f.write(f'element vertex {len(V)}\n')
            f.write('property float x\nproperty float y\nproperty float z\n')
            f.write(f'element face {len(F)}\n')
            f.write('property list uchar int vertex_indices\nend_header\n')
            for v in V: f.write(f'{v[0]:.4f} {v[1]:.4f} {v[2]:.4f}\n')
            for fc in F: f.write(f'3 {fc[0]} {fc[1]} {fc[2]}\n')
    kb = os.path.getsize(fpath)/1024
    print(f'  {fname:<30} {len(V):>8,} verts  {len(F):>8,} faces  {kb:>7.0f} KB')
    mat_plys[(role, mat)] = 'meshes/' + fname
    total_verts += len(V)
    total_faces += len(F)

print(f'Total : {total_verts:,} verts  {total_faces:,} faces')
print(f'Files : {len(mat_plys)} building PLYs + 1 terrain = {len(mat_plys)+1} shapes in scene.xml')


In [ ]:
# ============================================================
# CELL 4b - META AI CANOPY HEIGHT MODEL -> TREE PLYs
# ============================================================
# Downloads Meta/WRI CHM tiles (Web Mercator quadkey zoom-9).
# Uses windowed reads (clip to scene bbox) to avoid OOM.
# Vectorised numpy sampling — no per-point Python loop.

if not globals().get('USE_CANOPY_HEIGHTS', False):
    print('USE_CANOPY_HEIGHTS=False — skipping tree geometry')
else:
    import math, requests
    import numpy as np

    CHM_S3_BASE = 'https://dataforgood-fb-data.s3.amazonaws.com/forests/v1/alsgedi_global_v6_float/chm'
    R_EARTH = 6378137.0
    CHM_ZOOM = 9

    def _latlon_to_quadkey(lon, lat, zoom=CHM_ZOOM):
        extent = math.pi * R_EARTH
        x = lon * math.pi * R_EARTH / 180.0
        y = math.log(math.tan(math.pi / 4 + lat * math.pi / 360.0)) * R_EARTH
        u = (x + extent) / (2 * extent)
        v = (extent - y) / (2 * extent)
        n = 2 ** zoom
        tx = min(int(u * n), n - 1)
        ty = min(int(v * n), n - 1)
        qk = ''
        for i in range(zoom, 0, -1):
            bit = 1 << (i - 1)
            d = 0
            if tx & bit: d += 1
            if ty & bit: d += 2
            qk += str(d)
        return qk, tx, ty

    # Unique quadkey tiles covering scene bbox
    _corners = [(SCENE_WEST, SCENE_SOUTH), (SCENE_EAST, SCENE_SOUTH),
                (SCENE_WEST, SCENE_NORTH), (SCENE_EAST, SCENE_NORTH)]
    _needed_qk = {}
    for _lon_c, _lat_c in _corners:
        _qk, _tx, _ty = _latlon_to_quadkey(_lon_c, _lat_c)
        _needed_qk[_qk] = (_tx, _ty)
    print(f'Meta CHM quadkey tiles needed: {list(_needed_qk.keys())}')

    try:
        import rasterio
        from rasterio.io import MemoryFile
        from rasterio.windows import from_bounds as win_from_bounds, Window
        import pyproj
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, '-m', 'pip', 'install', 'rasterio', 'pyproj'], check=True)
        import rasterio
        from rasterio.io import MemoryFile
        from rasterio.windows import from_bounds as win_from_bounds, Window
        import pyproj

    # Scene bbox in Web Mercator for windowed reads
    _to_merc_bbox = pyproj.Transformer.from_crs('EPSG:4326', 'EPSG:3857', always_xy=True)
    _bbox_mx_w, _bbox_my_s = _to_merc_bbox.transform(SCENE_WEST,  SCENE_SOUTH)
    _bbox_mx_e, _bbox_my_n = _to_merc_bbox.transform(SCENE_EAST,  SCENE_NORTH)
    _MARGIN_M = 200
    _bbox_mx_w -= _MARGIN_M; _bbox_mx_e += _MARGIN_M
    _bbox_my_s -= _MARGIN_M; _bbox_my_n += _MARGIN_M
    print(f'Scene bbox (Mercator): x=[{_bbox_mx_w:.0f},{_bbox_mx_e:.0f}]  y=[{_bbox_my_s:.0f},{_bbox_my_n:.0f}]')

    # Download and clip tiles — store as (array, transform) strips
    _strips = []   # list of (np.ndarray float32, affine.Affine)

    for _qk in _needed_qk:
        _url = f'{CHM_S3_BASE}/{_qk}.tif'
        try:
            _r = requests.get(_url, timeout=120)
            if _r.status_code != 200:
                print(f'  Tile {_qk}: HTTP {_r.status_code} — skipping')
                continue
            with MemoryFile(_r.content) as _mf:
                with _mf.open() as _ds:
                    _win = win_from_bounds(
                        left=_bbox_mx_w, bottom=_bbox_my_s,
                        right=_bbox_mx_e, top=_bbox_my_n,
                        transform=_ds.transform,
                    ).round_offsets().round_lengths()
                    _ro = max(0, int(_win.row_off))
                    _co = max(0, int(_win.col_off))
                    _re = min(_ds.height, int(_win.row_off + _win.height))
                    _ce = min(_ds.width,  int(_win.col_off + _win.width))
                    if _re <= _ro or _ce <= _co:
                        print(f'  Tile {_qk}: no overlap — skipping')
                        continue
                    _clip_win   = Window(_co, _ro, _ce - _co, _re - _ro)
                    _clip_arr   = _ds.read(1, window=_clip_win).astype('float32')
                    _clip_trans = _ds.window_transform(_clip_win)
                    _nodata     = _ds.nodata
            # Zero out nodata
            if _nodata is not None:
                _clip_arr[_clip_arr == _nodata] = 0.0
            _valid = _clip_arr[_clip_arr > 0]
            _vmin  = float(_valid.min()) if len(_valid) else 0.0
            print(f'  Tile {_qk}: clipped {_clip_arr.shape}  range=[{_vmin:.1f},{float(_clip_arr.max()):.1f}] m')
            _strips.append((_clip_arr, _clip_trans))
        except Exception as _e:
            print(f'  Tile {_qk}: {_e}')

    if not _strips:
        print('No CHM tiles downloaded — skipping tree geometry')
    else:
        # ── Build a sample grid in Mercator coords directly ───────────────────
        # Sample at TREE_GRID_SPACING_M intervals across scene bbox
        _SPACING = TREE_GRID_SPACING_M
        _px_size = _strips[0][1].a   # pixel size (metres/px)

        _mx_grid = np.arange(_bbox_mx_w + _MARGIN_M, _bbox_mx_e - _MARGIN_M, _SPACING)
        _my_grid = np.arange(_bbox_my_s + _MARGIN_M, _bbox_my_n - _MARGIN_M, _SPACING)
        _MX, _MY = np.meshgrid(_mx_grid, _my_grid)   # shape (ny, nx)
        _MX = _MX.ravel(); _MY = _MY.ravel()
        print(f'Grid: {len(_mx_grid)} x {len(_my_grid)} = {len(_MX):,} points  (spacing={_SPACING}m)')

        # Sample each strip and take the maximum (handles overlap)
        _H_out = np.zeros(len(_MX), dtype='float32')

        for _arr, _tr in _strips:
            _cols = ((_MX - _tr.c) / _tr.a).astype(int)
            _rows = ((_MY - _tr.f) / _tr.e).astype(int)
            _mask = ((_rows >= 0) & (_rows < _arr.shape[0]) &
                     (_cols >= 0) & (_cols < _arr.shape[1]))
            _idx  = np.where(_mask)[0]
            if len(_idx) == 0: continue
            _h = _arr[_rows[_idx], _cols[_idx]]
            # Only overwrite where this strip has a larger (more valid) value
            _better = _h > _H_out[_idx]
            _H_out[_idx[_better]] = _h[_better]

        del _strips

        # Filter to tree height range
        _MIN_H, _MAX_H = TREE_MIN_HEIGHT_M, TREE_MAX_HEIGHT_M
        _tree_mask = (_H_out >= _MIN_H) & (_H_out <= _MAX_H)
        _tree_mx   = _MX[_tree_mask]
        _tree_my   = _MY[_tree_mask]
        _tree_h    = _H_out[_tree_mask]
        del _MX, _MY, _H_out

        print(f'Tree pixels in height range [{_MIN_H},{_MAX_H}] m: {len(_tree_h):,}')

        # Convert Mercator -> WGS84 -> local XYZ
        _from_merc = pyproj.Transformer.from_crs('EPSG:3857', 'EPSG:4326', always_xy=True)
        _utm_proj  = pyproj.Proj(f'epsg:{UTM_EPSG}')

        _tree_lon, _tree_lat = _from_merc.transform(_tree_mx, _tree_my)
        _tree_e,   _tree_n   = _utm_proj(_tree_lon, _tree_lat)
        _cx_utm, _cy_utm = _utm_proj(center_lon, center_lat)
        _tree_lx = _tree_e - _cx_utm
        _tree_ly = _tree_n - _cy_utm

        # Terrain Z at each tree (vectorised via DEM mosaic)
        # Use the height_from_wgs84 function defined in CELL 2
        _tree_z0 = np.array([local_z(float(_lo), float(_la))
                              for _lo, _la in zip(_tree_lon, _tree_lat)], dtype='float32')

        print(f'Generating {len(_tree_h):,} tree cylinders ...')

        _RADIUS = TREE_RADIUS_M
        _SIDES  = 8
        _angles = np.linspace(0, 2*math.pi, _SIDES, endpoint=False)
        _cos_a  = np.cos(_angles); _sin_a = np.sin(_angles)

        _tree_verts = []
        _tree_faces = []
        _voff = 0

        for _i in range(len(_tree_h)):
            _lx = float(_tree_lx[_i]); _ly = float(_tree_ly[_i])
            _z0 = float(_tree_z0[_i]); _h  = float(_tree_h[_i])
            _base = _voff
            for _s in range(_SIDES):
                _rx = _RADIUS * _cos_a[_s]; _ry = _RADIUS * _sin_a[_s]
                _tree_verts.append((_lx+_rx, _ly+_ry, _z0))
                _tree_verts.append((_lx+_rx, _ly+_ry, _z0+_h))
            for _s in range(_SIDES):
                _a0 = _base + 2*_s
                _a1 = _base + 2*((_s+1) % _SIDES)
                _tree_faces.append((_a0, _a1, _a0+1))
                _tree_faces.append((_a1, _a1+1, _a0+1))
            _voff += 2*_SIDES

        print(f'Placed {len(_tree_h):,} trees  ({len(_tree_verts):,} verts, {len(_tree_faces):,} faces)')

        if len(_tree_h) > 0:
            _ply_path = os.path.join(MESH_DIR, 'trees.ply')
            with open(_ply_path, 'w') as _f:
                _f.write('ply\nformat ascii 1.0\n')
                _f.write(f'element vertex {len(_tree_verts)}\n')
                _f.write('property float x\nproperty float y\nproperty float z\n')
                _f.write(f'element face {len(_tree_faces)}\n')
                _f.write('property list uchar int vertex_indices\n')
                _f.write('end_header\n')
                for _v in _tree_verts:
                    _f.write(f'{_v[0]:.3f} {_v[1]:.3f} {_v[2]:.3f}\n')
                for _fc in _tree_faces:
                    _f.write(f'3 {_fc[0]} {_fc[1]} {_fc[2]}\n')
            _ply_kb = os.path.getsize(_ply_path) / 1024
            print(f'Saved: {_ply_path}  ({_ply_kb:.0f} KB)')
            if 'mat_plys' not in dir():
                mat_plys = {}
            mat_plys[('trees', TREE_MATERIAL)] = 'meshes/trees.ply'
            print(f'Tree material: {TREE_MATERIAL}')
        else:
            print('No trees placed (no CHM values in valid range)')


In [ ]:
# ============================================================
# CELL 5 - WRITE SCENE.XML  (Mitsuba 2.1.0 / Sionna 0.19)
# ============================================================
ITU_MATERIALS = {
    'itu_concrete'   : (5.31,  0.092),
    'itu_brick'      : (3.75,  0.038),
    'itu_glass'      : (6.27,  0.000),
    'itu_wood'       : (1.99,  0.000),
    'itu_metal'      : (1.00,  1.0e7),
    'itu_asphalt'    : (2.56,  0.000),
    'itu_wet_ground' : (30.0,  0.020),
    # Custom absorbing urban ground — eps=5 (concrete-like), sigma=0.50
    # (high conductivity → strong absorption, breaks specular ground bounces
    #  that cause ~10 dB flat excess loss at all distances on flat terrain)
    'itu_urban_ground': (5.0,  0.50),
    # ITU-R P.833 vegetation: eps=1.0 (low permittivity), sigma=0.10
    # Acts as partial absorber — appropriate for tree canopy at 3.6 GHz
    'itu_vegetation'  : (1.0,  0.10),
}
# TERRAIN_MATERIAL is set in CELL 0

used_mats = set(mat for _, mat in mat_plys.keys()) | {TERRAIN_MATERIAL}

lines = []
lines.append('<?xml version="1.0" encoding="utf-8"?>')
lines.append('<scene version="2.1.0">')
lines.append('')
lines.append('  <!-- ITU-R P.2040-2 Materials -->')
for mat_name, (eps, sigma) in ITU_MATERIALS.items():
    if mat_name not in used_mats:
        continue
    lines.append(f'  <bsdf type="conductor" id="{mat_name}">')
    lines.append(f'    <float name="eta" value="{eps}"/>')
    lines.append(f'    <float name="k"   value="{sigma}"/>')
    lines.append(f'  </bsdf>')
    lines.append('')

lines.append('  <!-- Terrain -->')
lines.append('  <shape type="ply">')
lines.append('    <string name="filename" value="meshes/terrain.ply"/>')
lines.append(f'    <ref id="{TERRAIN_MATERIAL}" name="bsdf"/>')
lines.append('  </shape>')
lines.append('')

lines.append('  <!-- Buildings merged by material -->')
for (role, mat), ply_path in sorted(mat_plys.items()):
    lines.append(f'  <shape type="ply">  <!-- {role} -->')
    lines.append(f'    <string name="filename" value="{ply_path}"/>')
    lines.append(f'    <ref id="{mat}" name="bsdf"/>')
    lines.append(f'  </shape>')

lines.append('')
lines.append('</scene>')

scene_xml = os.path.join(SCENE_DIR, 'scene.xml')
with open(scene_xml, 'w') as f:
    f.write('\n'.join(lines))

xml_kb = os.path.getsize(scene_xml)/1024
print(f'Wrote: {scene_xml}  ({xml_kb:.1f} KB)')
print(f'  Shapes    : {1 + len(mat_plys)}  (1 terrain + {len(mat_plys)} building groups)')
print(f'  Materials : {len(used_mats)}')

meta = {
    'scene_center_lon'  : center_lon,
    'scene_center_lat'  : center_lat,
    'origin_elev_asl_m' : origin_elev_asl,
    'utm_epsg'          : UTM_EPSG,
    'bbox'              : {'west': SCENE_WEST, 'east': SCENE_EAST,
                           'south': SCENE_SOUTH, 'north': SCENE_NORTH},
    'n_buildings'       : n_ok,
    'terrain_grid_n'    : TERRAIN_GRID_N,
    'tile_zoom'         : TILE_ZOOM,
}
params_json = os.path.join(BASE_DIR, 'scene_parameters.json')
with open(params_json, 'w') as f:
    json.dump(meta, f, indent=2)
print(f'  Metadata  : {params_json}')

# Write origin_elev2.json in the format expected by sionna019_main_simulation.ipynb
# so FORCE_REBUILD_SCENE=False correctly restores the scene centre elevation.
_origin_json = os.path.join(SCENE_DIR, 'origin_elev2.json')
with open(_origin_json, 'w') as f:
    json.dump({'origin_elev2_m': origin_elev_asl}, f)
print(f'  origin_elev2.json : {_origin_json}  ({origin_elev_asl:.2f} m ASL)')


In [ ]:
# ============================================================
# CELL 6 — VERIFY SCENE
# ============================================================
# Quick sanity checks before handing the scene to the main notebook.

import glob as glob_mod

ply_files = sorted(glob_mod.glob(os.path.join(MESH_DIR, '*.ply')))
total_kb = sum(os.path.getsize(p) for p in ply_files) / 1024

print('=' * 60)
print('SCENE VERIFICATION')
print('=' * 60)
print(f'PLY files   : {len(ply_files)}')
print(f'Total size  : {total_kb/1024:.1f} MB')
print()

print(f'scene.xml   : {os.path.getsize(scene_xml)/1024:.0f} KB')

# Load with Mitsuba to verify (requires Sionna env)
try:
    import mitsuba as mi
    mi.set_variant('scalar_rgb')
    scene_mi = mi.load_file(scene_xml)
    bbox = scene_mi.bbox()
    print()
    print(f'Mitsuba load: OK')
    print(f'  BBox X    : [{float(bbox.min[0]):.1f}, {float(bbox.max[0]):.1f}] m')
    print(f'  BBox Y    : [{float(bbox.min[1]):.1f}, {float(bbox.max[1]):.1f}] m')
    print(f'  BBox Z    : [{float(bbox.min[2]):.1f}, {float(bbox.max[2]):.1f}] m')
except Exception as e:
    print(f'Mitsuba load: {e}')

print()
print('Scene metadata (scene_parameters.json):')
with open(params_json) as f:
    print(json.dumps(json.load(f), indent=2))

print()
print('DONE — scene ready for sionna019_main_simulation.ipynb')
print(f'Set BASE_DIR = "{BASE_DIR}" in Cell 0c of the main notebook.')